In [6]:
%pip install pygame
%pip install Pillow
%pip install requests beautifulsoup4
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import tkinter as tk  # 修正這一行
from tkinter import messagebox
from PIL import Image, ImageTk
import random
import pygame
import time
import os
import sys

# ... 後面代碼保持不變 ...

# ==========================================
#          📦 資源路徑處理 (For PyInstaller)
# ==========================================
def resource_path(relative_path):
    """ 獲取資源絕對路徑，兼容開發環境與 PyInstaller 打包環境 """
    try:
        base_path = sys._MEIPASS
    except Exception:
        base_path = os.path.abspath(".")
    return os.path.join(base_path, relative_path)

# ==========================================
#          📦 遊戲自定義設置 (Customization)
# ==========================================
WORD_BANK = ['IMPERIUM', 'HERESY', 'LOYALTY', 'SACRIFICE', 'CODEX', 'SERVITOR', 
             'ASTARTES', 'INQUISITOR','CUSTODES','MILITARUM','LOGISTICS',
             'BOLTER','TECHNOLOGY','WAR','CRUELTY','GRIM']
VICTIM_NAME = "Averyl Jingwei"
SPRITE_FILE = resource_path("boltgun.png") 

SOUNDS = {
    "correct": resource_path("correct.wav"),
    "load": resource_path("load_bolt.wav"),
    "bolt": resource_path("bolt_rack.wav"),
    "aim": resource_path("servo_aim.wav"),
    "safety": resource_path("click.wav"),
    "eat_bolt": resource_path("eat_boltgun.wav"),
    "fire": resource_path("bolter_shot.wav"),
    "eject": resource_path("shell_eject.wav")
}

STAGES = [
    "1. 處刑人取出了一枚 .75 口徑爆彈...",
    "2. 彈藥壓入彈匣。機魂正在甦醒。",
    "3. 槍機上膛。金屬碰撞聲在審訊室迴盪。",
    "4. 爆彈槍已對準罪人的頭顱。",
    "5. 保險關閉。手指已扣在扳機上。",
    "6. [ 已淨化 ]"
]

NARRATIVES = [
    "知識的代價是血。只有正確的供詞能救你。",
    "帝皇保佑那些言辭準確的人。再試一次。",
    "一個錯字，一個靈魂的隕落。聽聽那機魂的咆哮。",
    "沈默並不能保護你，只有真理可以。",
    "你是忠誠的僕人，還是隱藏的異端？"
]

class BolterTrial:
    def __init__(self, root):
        self.root = root
        self.root.title("審判庭：爆彈槍審訊系統")
        self.root.geometry("600x850")
        self.root.configure(bg="#050505")

        pygame.mixer.init()

        self.word = random.choice(WORD_BANK).upper()
        self.guessed_letters = set()
        self.mistakes = 0
        self.max_mistakes = len(STAGES) - 1

        try:
            img = Image.open(SPRITE_FILE)
            img = img.resize((250, 150), Image.Resampling.LANCZOS)
            self.boltgun_sprite = ImageTk.PhotoImage(img)
        except Exception as e:
            self.boltgun_sprite = None

        self.canvas = tk.Canvas(root, width=500, height=350, bg="#111", highlightthickness=2, highlightbackground="#444")
        self.canvas.pack(pady=20)
        
        self.status_label = tk.Label(root, text="--- 審訊開始：請輸入證詞 ---", fg="#ffcc00", bg="#050505", font=("Microsoft JhengHei", 12, "bold"))
        self.status_label.pack()

        self.word_display = tk.Label(root, text=self.get_display_word(), fg="#fff", bg="#050505", font=("Courier", 40, "bold"))
        self.word_display.pack(pady=20)

        self.narrative_display = tk.Label(root, text=NARRATIVES[0], fg="#888", bg="#050505", font=("Microsoft JhengHei", 10, "italic"), wraplength=500)
        self.narrative_display.pack(pady=10)

        self.entry = tk.Entry(root, font=("Courier", 22), width=5, justify='center', bg="#1a1a1a", fg="#ffcc00", insertbackground="white")
        self.entry.pack()
        self.entry.focus_set()
        self.entry.bind('<Return>', lambda e: self.make_guess())

        self.draw_trial()

    def screen_shake(self, intensity=20, duration=500):
        start_time = time.time() * 1000
        orig_x = self.root.winfo_x()
        orig_y = self.root.winfo_y()

        def shake():
            if (time.time() * 1000) - start_time < duration:
                dx = random.randint(-intensity, intensity)
                dy = random.randint(-intensity, intensity)
                self.root.geometry(f"+{orig_x + dx}+{orig_y + dy}")
                self.root.after(20, shake)
            else:
                self.root.geometry(f"+{orig_x}+{orig_y}")
        shake()

    def draw_explosion(self):
        """ 在受刑人頭部位置繪製紅橘色閃爆效果 """
        colors = ["#FF4500", "#FF8C00", "#FFD700", "#FFFFFF"] # 紅, 橘, 金, 白
        for _ in range(15):
            x = random.randint(310, 390)
            y = random.randint(90, 170)
            size = random.randint(10, 40)
            color = random.choice(colors)
            self.canvas.create_oval(x-size, y-size, x+size, y+size, fill=color, outline="")

    def play_sfx(self, sound_key):
        try:
            pygame.mixer.Sound(SOUNDS[sound_key]).play()
        except: pass

    def get_display_word(self):
        return " ".join([char if char in self.guessed_letters else "_" for char in self.word])

    def draw_trial(self):
        self.canvas.delete("all")
        self.canvas.create_oval(320, 100, 380, 160, outline="white", width=2) # 頭部
        self.canvas.create_line(350, 160, 350, 280, fill="white", width=2)    # 身體
        
        if self.mistakes >= 1:
            y_pos = 180 if self.mistakes < 4 else 140
            if self.boltgun_sprite:
                self.canvas.create_image(150, y_pos, image=self.boltgun_sprite)
            else:
                self.canvas.create_rectangle(50, y_pos-30, 250, y_pos+30, fill="#600", outline="#f00")
            
        if self.mistakes >= 4:
            self.canvas.create_line(230, 145, 320, 130, fill="red", width=2, dash=(5, 2))

    def make_guess(self):
        guess = self.entry.get().upper().strip()
        self.entry.delete(0, tk.END)
        if len(guess) != 1 or not guess.isalpha() or guess in self.guessed_letters: return

        self.guessed_letters.add(guess)
        if guess in self.word:
            self.play_sfx("correct")
            self.word_display.config(text=self.get_display_word())
            if "_" not in self.get_display_word(): self.end_game(True)
        else:
            self.mistakes += 1
            self.trigger_mistake_sequence()

    def trigger_mistake_sequence(self):
        self.status_label.config(text=STAGES[self.mistakes-1])
        self.narrative_display.config(text=random.choice(NARRATIVES))
        self.draw_trial()

        if self.mistakes < self.max_mistakes:
            s_map = {1: "load", 2: "load", 3: "bolt", 4: "aim", 5: "safety"}
            if self.mistakes in s_map: self.play_sfx(s_map[self.mistakes])
        else:
            self.play_sfx("eat_bolt")
            self.root.after(600, self.execute_fire)

    def execute_fire(self):
        self.play_sfx("fire")
        self.screen_shake(intensity=25)
        self.draw_explosion() # 繪製爆炸
        self.root.after(600, lambda: self.play_sfx("eject"))
        self.root.after(1500, lambda: self.end_game(False))

    def end_game(self, won):
        if won:
            messagebox.showinfo("審判結束", f"真理已明。{VICTIM_NAME} 被判無罪釋放。")
        else:
            msg = (f"{VICTIM_NAME} 在一場由拼寫錯誤引發的危機中喪生。\n"
                   f"[ 已淨化 ]\n"
                   f"我們將永遠記住知識、進步與錯誤的代價。\n"
                   f"正確的單字是: {self.word}")
            messagebox.showerror("帝皇的裁決", msg)
        self.root.destroy()

if __name__ == "__main__":
    root = tk.Tk()
    game = BolterTrial(root)
    root.mainloop()

c:\Users\kingo\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
import tkinter as tk
from tkinter import messagebox, scrolledtext
from PIL import Image, ImageTk
import random
import pygame
import time
import os
import sys
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt

# 設定中文顯示
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei'] 
plt.rcParams['axes.unicode_minus'] = False

def resource_path(relative_path):
    try:
        base_path = sys._MEIPASS
    except Exception:
        base_path = os.path.abspath(".")
    return os.path.join(base_path, relative_path)

# --- 爬蟲模組 ---
CONNECTION_STATUS = "Checking..."

def fetch_military_word_bank():
    global CONNECTION_STATUS
    url = "https://en.wikipedia.org/wiki/List_of_established_military_terms"
    backup_words = ['IMPERIUM', 'HERESY', 'LOYALTY', 'SACRIFICE', 'CODEX', 'BOLTER', 'WAR']
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=5)
        soup = BeautifulSoup(response.text, 'html.parser')
        word_pool = []
        for item in soup.find_all('li'):
            bold_text = item.find('b')
            raw_text = bold_text.get_text().strip() if bold_text else item.get_text().split(':')[0].strip()
            if raw_text.isalpha() and 4 <= len(raw_text) <= 12:
                word_pool.append(raw_text.upper())
        
        if len(word_pool) > 10:
            CONNECTION_STATUS = "● Online (Wiki API)"
            return list(set(word_pool))
        else:
            CONNECTION_STATUS = "○ Local (Low Results)"
            return backup_words
    except:
        CONNECTION_STATUS = "○ Offline (Local DB)"
        return backup_words

WORD_BANK = fetch_military_word_bank()

class BolterTrial:
    def __init__(self, root):
        self.root = root
        self.root.title("審判庭：爆彈槍審訊系統")
        self.root.geometry("600x950")
        self.root.configure(bg="#050505")
        
        pygame.mixer.init()
        self.game_history = [] 
        self.dev_mode = False 
        
        try:
            img = Image.open(resource_path("boltgun.png"))
            img = img.resize((250, 150), Image.Resampling.LANCZOS)
            self.boltgun_sprite = ImageTk.PhotoImage(img)
        except:
            self.boltgun_sprite = None

        # --- DevMode 介面元件 ---
        self.dev_btn = tk.Button(root, text="DEV", command=self.toggle_dev_mode, 
                                bg="#222", fg="#444", font=("Arial", 8), 
                                bd=0, activebackground="#333")
        self.dev_btn.place(x=5, y=5)
        
        self.dev_info_frame = tk.Frame(root, bg="#050505")
        self.dev_label_word = tk.Label(self.dev_info_frame, text="", fg="#00FF00", bg="#050505", font=("Consolas", 9))
        self.dev_label_conn = tk.Label(self.dev_info_frame, text=f"Network: {CONNECTION_STATUS}", 
                                      fg="#00AAAA", bg="#050505", font=("Consolas", 9))

        # --- 核心遊戲 UI ---
        # 修正佈局：pady=(70, 10) 確保上方預留 70 像素空間給 Dev 資訊
        self.canvas = tk.Canvas(root, width=500, height=350, bg="#111", highlightthickness=2, highlightbackground="#444")
        self.canvas.pack(pady=(70, 10)) 
        
        self.status_label = tk.Label(root, text="", fg="#ffcc00", bg="#050505", font=("Microsoft JhengHei", 12, "bold"))
        self.status_label.pack()

        self.word_display = tk.Label(root, text="", fg="#fff", bg="#050505", font=("Courier", 40, "bold"))
        self.word_display.pack(pady=10)

        self.narrative_display = tk.Label(root, text="", fg="#888", bg="#050505", font=("Microsoft JhengHei", 10, "italic"), wraplength=500)
        self.narrative_display.pack(pady=5)

        self.entry = tk.Entry(root, font=("Courier", 22), width=5, justify='center', bg="#1a1a1a", fg="#ffcc00", insertbackground="white")
        self.entry.pack(pady=10)
        self.entry.bind('<Return>', lambda e: self.make_guess())

        self.btn_frame = tk.Frame(root, bg="#050505")
        self.btn_frame.pack(pady=20)
        tk.Button(self.btn_frame, text="查看戰績紀錄", command=self.show_history, bg="#333", fg="white").grid(row=0, column=0, padx=10)
        tk.Button(self.btn_frame, text="結算並退出", command=self.finalize_and_exit, bg="#600", fg="white").grid(row=0, column=1, padx=10)

        self.setup_game()

    def toggle_dev_mode(self):
        """ 切換開發者模式顯示，並修正層級遮擋 """
        self.dev_mode = not self.dev_mode
        if self.dev_mode:
            self.dev_btn.config(fg="#00FF00")
            # 調整 y=28 使其完美顯示於按鈕下方
            self.dev_info_frame.place(x=5, y=28) 
            self.dev_label_word.config(text=f"Solution: {self.word}")
            self.dev_label_word.pack(anchor="w")
            self.dev_label_conn.pack(anchor="w")
            # 強制提升層級，防止 Canvas 覆蓋
            self.dev_info_frame.lift() 
        else:
            self.dev_btn.config(fg="#444")
            self.dev_info_frame.place_forget()

    def play_sfx(self, sound_key):
        sounds = {
            "correct": "correct.wav", "load": "load_bolt.wav", "bolt": "bolt_rack.wav",
            "aim": "servo_aim.wav", "safety": "click.wav", "eat_bolt": "eat_boltgun.wav",
            "fire": "bolter_shot.wav", "eject": "shell_eject.wav"
        }
        try: pygame.mixer.Sound(resource_path(sounds[sound_key])).play()
        except: pass

    def setup_game(self):
        self.word = random.choice(WORD_BANK).upper()
        self.guessed_letters = set()
        self.mistakes = 0
        self.max_mistakes = 5
        self.guess_count = 0
        
        if self.dev_mode:
            self.dev_label_word.config(text=f"Solution: {self.word}")
            self.dev_info_frame.lift()
            
        self.entry.config(state='normal')
        self.entry.delete(0, tk.END)
        self.entry.focus_set()
        self.status_label.config(text="--- 審訊重啟：請輸入證詞 ---")
        self.word_display.config(text=self.get_display_word())
        self.narrative_display.config(text="帝皇正在注視著你...")
        self.draw_trial()

    def get_display_word(self):
        return " ".join([char if char in self.guessed_letters else "_" for char in self.word])

    def draw_trial(self):
        self.canvas.delete("all")
        self.canvas.create_oval(320, 100, 380, 160, outline="white", width=2)
        self.canvas.create_line(350, 160, 350, 280, fill="white", width=2)
        if self.mistakes >= 1:
            y_pos = 180 if self.mistakes < 4 else 140
            if self.boltgun_sprite:
                self.canvas.create_image(150, y_pos, image=self.boltgun_sprite)
            else:
                self.canvas.create_rectangle(50, y_pos-30, 250, y_pos+30, fill="#600")
        if self.mistakes >= 4:
            self.canvas.create_line(230, 145, 320, 130, fill="red", width=2, dash=(5, 2))

    def make_guess(self):
        guess = self.entry.get().upper().strip()
        self.entry.delete(0, tk.END)
        if len(guess) != 1 or not guess.isalpha() or guess in self.guessed_letters: return
        self.guess_count += 1
        self.guessed_letters.add(guess)
        if guess in self.word:
            self.play_sfx("correct")
            self.word_display.config(text=self.get_display_word())
            if "_" not in self.get_display_word(): self.handle_end(True)
        else:
            self.mistakes += 1
            self.trigger_mistake_sequence()

    def trigger_mistake_sequence(self):
        stages_text = ["1. 取出爆彈", "2. 裝填彈藥", "3. 槍機上膛", "4. 瞄準目標", "5. 解除保險", "6. [ 已淨化 ]"]
        self.status_label.config(text=stages_text[self.mistakes-1])
        self.draw_trial()
        if self.mistakes < self.max_mistakes:
            s_map = {1: "load", 2: "load", 3: "bolt", 4: "aim", 5: "safety"}
            self.play_sfx(s_map.get(self.mistakes))
        else:
            self.entry.config(state='disabled')
            self.play_sfx("eat_bolt")
            self.root.after(600, self.execute_fire_sequence)

    def execute_fire_sequence(self):
        self.play_sfx("fire")
        self.screen_shake(intensity=30)
        self.draw_explosion()
        self.root.after(600, lambda: self.play_sfx("eject"))
        self.root.after(1500, lambda: self.handle_end(False))

    def screen_shake(self, intensity=25, duration=500):
        start_time = time.time() * 1000
        orig_x, orig_y = self.root.winfo_x(), self.root.winfo_y()
        def shake():
            if (time.time() * 1000) - start_time < duration:
                dx, dy = random.randint(-intensity, intensity), random.randint(-intensity, intensity)
                self.root.geometry(f"+{orig_x + dx}+{orig_y + dy}")
                self.root.after(20, shake)
            else: self.root.geometry(f"+{orig_x}+{orig_y}")
        shake()

    def draw_explosion(self):
        colors = ["#FF4500", "#FF8C00", "#FFD700", "#FFFFFF"]
        for _ in range(20):
            x, y = random.randint(310, 390), random.randint(90, 170)
            s = random.randint(10, 40)
            self.canvas.create_oval(x-s, y-s, x+s, y+s, fill=random.choice(colors), outline="")

    def handle_end(self, won):
        result_str = "勝訴" if won else "淨化"
        self.game_history.append({"word": self.word, "result": result_str, "guesses": self.guess_count})
        msg = "真理已明，是否繼續？" if won else "異端已滅，是否繼續審判？"
        if messagebox.askyesno("審訊報告", msg): self.setup_game()
        else: self.finalize_and_exit()

    def show_history(self):
        win = tk.Toplevel(self.root)
        win.title("審判庭存檔")
        txt = scrolledtext.ScrolledText(win, width=45, height=20, bg="#111", fg="#eee")
        txt.pack()
        txt.insert(tk.END, "--- 歷史紀錄 ---\n")
        for i, h in enumerate(self.game_history, 1):
            txt.insert(tk.END, f"{i:02d}. [{h['word']}] -> {h['result']}\n")
        txt.config(state='disabled')

    def finalize_and_exit(self):
        if self.game_history:
            wins = sum(1 for h in self.game_history if h['result'] == "勝訴")
            losses = len(self.game_history) - wins
            plt.figure(figsize=(6, 5))
            plt.bar(['忠誠 (勝)', '異端 (敗)'], [wins, losses], color=['#4CAF50', '#F44336'])
            plt.title(f"總計遊玩 {len(self.game_history)} 局")
            plt.show()
        self.root.destroy()

if __name__ == "__main__":
    root = tk.Tk()
    game = BolterTrial(root)
    root.mainloop()